# 🚀 Fine-Tune Qwen2.5-VL-7B for POD (Proof of Delivery) Analysis

This notebook fine-tunes **Qwen2.5-VL-7B-Instruct** using **Unsloth + QLoRA (4-bit)** to analyze POD images and output structured JSON.

## Requirements
- **Kaggle GPU**: T4 x2 (free tier) — select from Settings → Accelerator
- **Dataset**: Upload `kaggle_dataset_upload` as a Kaggle Dataset
- **Internet**: Enable Internet access in notebook settings
- **HuggingFace Token**: Add as Kaggle Secret named `HF_TOKEN` (optional, for pushing to Hub)

## What the model learns
Given a POD image, output:
```json
{
  "cnNumber": "string or null",
  "hasSignature": true,
  "hasStamp": true,
  "hasHandwriting": true,
  "imageQualityPassed": true,
  "remarksText": "string or null",
  "deliveryDate": "YYYY-MM-DD or null",
  "categoryReason": "string",
  "confidenceScore": 0.95,
  "podCategory": "CLEAN_POD_SEAL_AND_SIGNATURE",
  "limit_exceed": false
}
```

---
## 📦 Step 1: Install Dependencies
This takes ~3-5 minutes on Kaggle. We use `%%capture` to keep output clean.

In [1]:
%%capture
!pip install --upgrade pip
!pip install unsloth
!pip install --no-deps xformers trl peft accelerate bitsandbytes
!pip install qwen-vl-utils
print("✅ All dependencies installed!")

In [2]:
# Verify GPU availability
import torch
print(f"PyTorch version: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)} — {torch.cuda.get_device_properties(i).total_mem / 1e9:.1f} GB")
    print(f"  BF16 support: {torch.cuda.is_bf16_supported()}")
else:
    raise RuntimeError("❌ No GPU found! Enable GPU in Kaggle Settings → Accelerator → GPU T4 x2")

PyTorch version: 2.10.0+cpu
CUDA available: False


RuntimeError: ❌ No GPU found! Enable GPU in Kaggle Settings → Accelerator → GPU T4 x2

---
## 🧠 Step 2: Load Model (4-bit Quantized)
We use Unsloth's pre-quantized 4-bit version to fit in Kaggle's 16GB T4 VRAM.

In [ ]:
from unsloth import FastVisionModel
import torch

# ========================================================================
# MODEL CONFIGURATION
# ========================================================================
MODEL_NAME = "unsloth/Qwen2.5-VL-7B-Instruct-bnb-4bit"
MAX_SEQ_LENGTH = 2048   # Covers our JSON output length comfortably

model, tokenizer = FastVisionModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=True,                       # QLoRA 4-bit quantization
    use_gradient_checkpointing="unsloth",     # 60% less VRAM
)

print(f"✅ Model loaded: {MODEL_NAME}")
print(f"   Trainable params before LoRA: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

---
## 🔧 Step 3: Apply QLoRA (Parameter-Efficient Fine-Tuning)
We fine-tune both vision and language layers with LoRA rank 16.

In [ ]:
model = FastVisionModel.get_peft_model(
    model,
    # Fine-tune ALL components for best accuracy on our specific task
    finetune_vision_layers=True,       # Train the ViT encoder (image understanding)
    finetune_language_layers=True,      # Train the LLM decoder (JSON generation)
    finetune_attention_modules=True,    # Train attention heads
    finetune_mlp_modules=True,          # Train MLP/feed-forward layers
    
    # LoRA hyperparameters
    r=16,                  # LoRA rank — higher = more capacity, more VRAM
    lora_alpha=16,         # Scaling factor (alpha/r = scaling)
    lora_dropout=0,        # No dropout — Unsloth recommends 0 for LoRA
    bias="none",           # Don't train biases — saves memory
    use_rslora=False,      # Standard LoRA (not rank-stabilized)
    random_state=42,       # Reproducibility
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"✅ LoRA applied!")
print(f"   Trainable parameters: {trainable:,} ({100*trainable/total:.2f}% of {total:,})")

---
## 📊 Step 4: Prepare Dataset

We convert the JSONL training data into the format expected by Unsloth's `SFTTrainer`.
Each sample becomes a multi-turn conversation with:
- **System**: POD analysis rules
- **User**: Image + prompt  
- **Assistant**: Structured JSON output

In [ ]:
import json
import os
from PIL import Image
from datasets import Dataset

# ========================================================================
# DATASET PATH — Update this to match your Kaggle dataset name!
# When you upload "kaggle_dataset_upload" as a Kaggle dataset,
# it appears at: /kaggle/input/<your-dataset-name>/
# ========================================================================
DATASET_DIR = "/kaggle/input/kaggle-dataset-upload"  # <-- change if needed
JSONL_PATH = os.path.join(DATASET_DIR, "kaggle_dataset_qwen_vl.jsonl")

# Verify paths exist
assert os.path.exists(DATASET_DIR), f"❌ Dataset not found at {DATASET_DIR}. Check your Kaggle dataset name!"
assert os.path.exists(JSONL_PATH), f"❌ JSONL file not found at {JSONL_PATH}"

print(f"📂 Dataset directory: {DATASET_DIR}")
print(f"📄 JSONL file: {JSONL_PATH}")
print(f"📁 Files in dataset dir: {len(os.listdir(DATASET_DIR))}")

In [ ]:
# ========================================================================
# SYSTEM PROMPT — This defines what the model should do
# ========================================================================
SYSTEM_PROMPT = """You are a highly accurate document analysis AI specialized in processing Proof of Delivery (POD) images.
You must analyze the provided POD image and extract specific information.

Classification Rules (in priority order):
1. Physical Paper Damage (torn, ripped, missing sections) -> MANUAL_CHECK_REQUIRED
2. Damage + Short mentioned in remarks -> ISSUE_POD_DAMAGED_AND_SHORT
3. Damage mentioned in remarks -> ISSUE_POD_DAMAGED
4. Shortage mentioned in remarks -> ISSUE_POD_SHORT
5. Seal/Stamp + Signature present -> CLEAN_POD_SEAL_AND_SIGNATURE
6. Seal/Stamp Only -> CLEAN_POD_ONLY_SEAL
7. Signature Only -> CLEAN_POD_ONLY_SIGNATURE
8. Neither Seal nor Signature -> NO_SIGNATURE_NO_STAMP

Output ONLY valid JSON with these exact fields:
- cnNumber: string or null (consignment number)
- hasSignature: boolean
- hasStamp: boolean
- hasHandwriting: boolean
- imageQualityPassed: boolean
- remarksText: string or null
- deliveryDate: "YYYY-MM-DD" or null
- categoryReason: string (explain your classification)
- confidenceScore: float (0.0 to 1.0)
- podCategory: string (one of the categories above)
- limit_exceed: boolean (false for normal deliveries)"""

USER_PROMPT = "Extract all POD fields into JSON."


def add_confidence_and_limit(original_json_str):
    """
    Add 'confidenceScore' and 'limit_exceed' fields to the training JSON output
    if they are missing. Also remove extra fields not in our target schema.
    """
    try:
        data = json.loads(original_json_str)
    except json.JSONDecodeError:
        return original_json_str

    # Build output in exact target schema order
    output = {
        "cnNumber": data.get("cnNumber"),
        "hasSignature": data.get("hasSignature", False),
        "hasStamp": data.get("hasStamp", False),
        "hasHandwriting": data.get("hasHandwriting", False),
        "imageQualityPassed": data.get("imageQualityPassed", True),
        "remarksText": data.get("remarksText"),
        "deliveryDate": data.get("deliveryDate"),
        "categoryReason": data.get("categoryReason", ""),
        "confidenceScore": data.get("confidenceScore", 0.95),
        "podCategory": data.get("podCategory", ""),
        "limit_exceed": data.get("limit_exceed", False),
    }

    return json.dumps(output, indent=2)


def load_and_convert_dataset(jsonl_path, dataset_dir):
    """
    Load the Qwen-VL JSONL and convert to Unsloth's expected format.
    Each sample becomes a conversation with system/user/assistant messages.
    Images are loaded as PIL Image objects.
    """
    converted_samples = []
    skipped = 0

    with open(jsonl_path, "r", encoding="utf-8") as f:
        for line_num, line in enumerate(f, 1):
            line = line.strip()
            if not line:
                continue

            try:
                entry = json.loads(line)
            except json.JSONDecodeError:
                print(f"  ⚠️ Line {line_num}: Invalid JSON, skipping")
                skipped += 1
                continue

            messages = entry.get("messages", [])
            if len(messages) < 2:
                skipped += 1
                continue

            # Find the image path from user message
            image_path = None
            for msg in messages:
                if msg["role"] == "user" and isinstance(msg["content"], list):
                    for item in msg["content"]:
                        if isinstance(item, dict) and item.get("type") == "image":
                            image_path = item.get("image", "")
                            break

            if not image_path:
                skipped += 1
                continue

            # Build absolute image path
            abs_image_path = os.path.join(dataset_dir, image_path)
            if not os.path.exists(abs_image_path):
                print(f"  ⚠️ Line {line_num}: Image not found: {abs_image_path}")
                skipped += 1
                continue

            # Load the image
            try:
                img = Image.open(abs_image_path).convert("RGB")
                # Resize large images to save VRAM (max 1024px on longest side)
                max_dim = 1024
                if max(img.size) > max_dim:
                    ratio = max_dim / max(img.size)
                    new_size = (int(img.size[0] * ratio), int(img.size[1] * ratio))
                    img = img.resize(new_size, Image.LANCZOS)
            except Exception as e:
                print(f"  ⚠️ Line {line_num}: Error loading image: {e}")
                skipped += 1
                continue

            # Get assistant response and normalize it
            assistant_content = ""
            for msg in messages:
                if msg["role"] == "assistant":
                    assistant_content = msg["content"]
                    break

            # Normalize the JSON output to match our target schema
            normalized_output = add_confidence_and_limit(assistant_content)

            # Build the conversation in Unsloth format
            sample = {
                "messages": [
                    {
                        "role": "system",
                        "content": [{"type": "text", "text": SYSTEM_PROMPT}],
                    },
                    {
                        "role": "user",
                        "content": [
                            {"type": "image", "image": img},
                            {"type": "text", "text": USER_PROMPT},
                        ],
                    },
                    {
                        "role": "assistant",
                        "content": [{"type": "text", "text": normalized_output}],
                    },
                ],
            }
            converted_samples.append(sample)

    print(f"\n✅ Dataset loaded:")
    print(f"   Total samples: {len(converted_samples)}")
    print(f"   Skipped: {skipped}")
    return converted_samples


# Load and convert
dataset_samples = load_and_convert_dataset(JSONL_PATH, DATASET_DIR)

In [ ]:
# ========================================================================
# SPLIT INTO TRAIN / VALIDATION
# ========================================================================
import random
random.seed(42)

# Shuffle and split: 90% train, 10% validation
random.shuffle(dataset_samples)
split_idx = int(len(dataset_samples) * 0.9)

train_samples = dataset_samples[:split_idx]
val_samples = dataset_samples[split_idx:]

print(f"📊 Train samples: {len(train_samples)}")
print(f"📊 Val samples:   {len(val_samples)}")

# Show a sample to verify format
print(f"\n--- Sample training entry ---")
sample = train_samples[0]
print(f"System: {sample['messages'][0]['content'][0]['text'][:100]}...")
print(f"User text: {sample['messages'][1]['content'][1]['text']}")
print(f"User image: {type(sample['messages'][1]['content'][0]['image'])}")
print(f"Assistant output (first 200 chars): {sample['messages'][2]['content'][0]['text'][:200]}...")

In [ ]:
# ========================================================================
# ANALYZE CATEGORY DISTRIBUTION (helps verify data quality)
# ========================================================================
from collections import Counter

categories = []
for s in dataset_samples:
    try:
        output = json.loads(s["messages"][2]["content"][0]["text"])
        categories.append(output.get("podCategory", "UNKNOWN"))
    except:
        categories.append("PARSE_ERROR")

cat_counts = Counter(categories)
print("📊 POD Category Distribution:")
print("=" * 50)
for cat, count in cat_counts.most_common():
    bar = "█" * (count * 40 // max(cat_counts.values()))
    print(f"  {cat:<40s} {count:>3d} {bar}")

---
## 🏋️ Step 5: Training

We use `SFTTrainer` with Unsloth's optimized `UnslothVisionDataCollator` for:
- Automatic image preprocessing & tokenization
- Proper padding/attention masking for vision-language inputs
- Memory-efficient batching

### Key hyperparameters:
| Parameter | Value | Why |
|---|---|---|
| Batch size | 1 | T4 VRAM constraint |
| Gradient accumulation | 8 | Effective batch size = 8 |
| Learning rate | 1e-4 | Good for LoRA fine-tuning |
| Epochs | 3 | Small dataset = more epochs |
| Warmup ratio | 0.1 | 10% warmup for stable start |

In [ ]:
from unsloth.trainer import UnslothVisionDataCollator
from trl import SFTTrainer, SFTConfig

# Enable training mode
FastVisionModel.for_training(model)

# ========================================================================
# TRAINING CONFIGURATION
# ========================================================================
NUM_EPOCHS = 3
BATCH_SIZE = 1               # Max 1 for T4 with 7B VLM
GRAD_ACCUM = 8               # Effective batch size = 1 * 8 = 8
LEARNING_RATE = 1e-4         # Recommended for QLoRA
WARMUP_RATIO = 0.1           # 10% warmup
OUTPUT_DIR = "/kaggle/working/pod_model_output"

training_args = SFTConfig(
    output_dir=OUTPUT_DIR,
    
    # Core training params
    num_train_epochs=NUM_EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    
    # Learning rate schedule
    learning_rate=LEARNING_RATE,
    lr_scheduler_type="cosine",
    warmup_ratio=WARMUP_RATIO,
    
    # Precision — use BF16 if available, else FP16
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    
    # Optimization
    optim="adamw_8bit",       # 8-bit AdamW saves ~50% optimizer memory
    weight_decay=0.01,
    max_grad_norm=1.0,
    
    # Logging & Saving
    logging_steps=5,
    save_strategy="epoch",
    save_total_limit=2,
    eval_strategy="epoch",
    
    # Sequence length
    max_seq_length=MAX_SEQ_LENGTH,
    dataset_text_field="",           # Not used for vision
    dataset_kwargs={"skip_prepare_dataset": True},
    
    # Misc
    seed=42,
    report_to="none",         # No W&B — set to "wandb" if you want logging
    remove_unused_columns=False,
    dataloader_pin_memory=True,
)

# Create the trainer
trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    data_collator=UnslothVisionDataCollator(model, tokenizer),
    train_dataset=train_samples,
    eval_dataset=val_samples,
    args=training_args,
)

print(f"\n✅ Trainer ready!")
print(f"   Training samples: {len(train_samples)}")
print(f"   Validation samples: {len(val_samples)}")
print(f"   Total steps: {len(train_samples) * NUM_EPOCHS // (BATCH_SIZE * GRAD_ACCUM)}")
print(f"   Effective batch size: {BATCH_SIZE * GRAD_ACCUM}")

In [ ]:
# ========================================================================
# 🚀 START TRAINING
# ========================================================================
# On Kaggle T4x2 with 174 samples × 3 epochs, this takes ~30-60 minutes.

import time
import gc

# Clear GPU cache before training
gc.collect()
torch.cuda.empty_cache()

print(f"🏁 Training started at {time.strftime('%H:%M:%S')}")
print(f"   GPU memory allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB")
print(f"   GPU memory reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB")
print("=" * 60)

train_result = trainer.train()

print("=" * 60)
print(f"✅ Training completed at {time.strftime('%H:%M:%S')}!")
print(f"   Final train loss: {train_result.training_loss:.4f}")
print(f"   Total train time: {train_result.metrics['train_runtime']:.0f}s")
print(f"   Samples/sec: {train_result.metrics['train_samples_per_second']:.2f}")

In [ ]:
# ========================================================================
# PLOT TRAINING LOSS
# ========================================================================
import matplotlib.pyplot as plt

# Extract loss from training log
log_history = trainer.state.log_history
train_losses = [(entry["step"], entry["loss"]) for entry in log_history if "loss" in entry]
eval_losses = [(entry["step"], entry["eval_loss"]) for entry in log_history if "eval_loss" in entry]

fig, ax = plt.subplots(figsize=(10, 5))
if train_losses:
    steps, losses = zip(*train_losses)
    ax.plot(steps, losses, label="Train Loss", color="#2196F3", linewidth=2)
if eval_losses:
    steps, losses = zip(*eval_losses)
    ax.plot(steps, losses, label="Eval Loss", color="#FF5722", linewidth=2, marker="o")

ax.set_xlabel("Steps")
ax.set_ylabel("Loss")
ax.set_title("Qwen2.5-VL-7B POD Fine-Tuning Loss")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig("/kaggle/working/training_loss.png", dpi=150)
plt.show()
print("📈 Loss plot saved to /kaggle/working/training_loss.png")

---
## 🧪 Step 6: Evaluation & Inference

Test the fine-tuned model on validation samples to check JSON output quality.

In [ ]:
# ========================================================================
# INFERENCE HELPER FUNCTION
# ========================================================================
from qwen_vl_utils import process_vision_info

# Switch model to inference mode (2x faster generation)
FastVisionModel.for_inference(model)


def predict_pod(image_input, model=model, tokenizer=tokenizer):
    """
    Run inference on a single POD image.
    
    Args:
        image_input: PIL Image, file path string, or URL
    
    Returns:
        dict: Parsed JSON output or raw string if parsing fails
    """
    # Handle different input types
    if isinstance(image_input, str):
        image_content = {"type": "image", "image": image_input}
    else:
        image_content = {"type": "image", "image": image_input}
    
    messages = [
        {
            "role": "system",
            "content": [{"type": "text", "text": SYSTEM_PROMPT}],
        },
        {
            "role": "user",
            "content": [
                image_content,
                {"type": "text", "text": USER_PROMPT},
            ],
        },
    ]

    # Apply chat template
    input_text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=True
    )
    
    # Process images
    image_inputs, video_inputs = process_vision_info(messages)
    
    # Tokenize
    inputs = tokenizer(
        text=[input_text],
        images=image_inputs,
        videos=video_inputs,
        padding=True,
        return_tensors="pt",
    ).to(model.device)

    # Generate
    with torch.no_grad():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=512,
            temperature=0.1,      # Low temp for deterministic JSON output
            top_p=0.95,
            do_sample=True,
            repetition_penalty=1.1,
        )

    # Decode — skip the input tokens
    generated_ids = output_ids[:, inputs["input_ids"].shape[1]:]
    response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

    # Try to parse as JSON
    try:
        # Handle case where model outputs ```json ... ```
        clean = response.strip()
        if clean.startswith("```json"):
            clean = clean[7:]
        if clean.startswith("```"):
            clean = clean[3:]
        if clean.endswith("```"):
            clean = clean[:-3]
        return json.loads(clean.strip())
    except json.JSONDecodeError:
        return {"raw_output": response, "parse_error": True}


print("✅ Inference function ready!")

In [ ]:
# ========================================================================
# RUN EVALUATION ON VALIDATION SET
# ========================================================================
print("🧪 Running evaluation on validation set...")
print("=" * 70)

eval_results = []
correct_category = 0
valid_json_count = 0
total_evaluated = min(len(val_samples), 10)  # Evaluate up to 10 samples

for i, sample in enumerate(val_samples[:total_evaluated]):
    print(f"\n--- Sample {i+1}/{total_evaluated} ---")
    
    # Get the image from the sample
    image = sample["messages"][1]["content"][0]["image"]
    
    # Get expected output
    expected_text = sample["messages"][2]["content"][0]["text"]
    try:
        expected = json.loads(expected_text)
    except:
        expected = {}
    
    # Get prediction
    prediction = predict_pod(image)
    
    # Check quality
    is_valid_json = "parse_error" not in prediction
    if is_valid_json:
        valid_json_count += 1
    
    category_match = (
        is_valid_json 
        and prediction.get("podCategory") == expected.get("podCategory")
    )
    if category_match:
        correct_category += 1
    
    # Print comparison
    print(f"  Expected category:  {expected.get('podCategory', 'N/A')}")
    print(f"  Predicted category: {prediction.get('podCategory', 'PARSE_ERROR')}")
    print(f"  Category match: {'✅' if category_match else '❌'}")
    print(f"  Valid JSON: {'✅' if is_valid_json else '❌'}")
    
    if is_valid_json:
        # Check individual field matches
        fields_to_check = ["hasSignature", "hasStamp", "hasHandwriting"]
        for field in fields_to_check:
            match = prediction.get(field) == expected.get(field)
            print(f"    {field}: {prediction.get(field)} {'✅' if match else '❌ (expected: ' + str(expected.get(field)) + ')'}")
    
    eval_results.append({
        "expected": expected,
        "predicted": prediction,
        "category_match": category_match,
        "valid_json": is_valid_json,
    })
    
    # Free memory
    torch.cuda.empty_cache()

# Print summary
print("\n" + "=" * 70)
print(f"📊 EVALUATION SUMMARY ({total_evaluated} samples)")
print(f"   Valid JSON output:    {valid_json_count}/{total_evaluated} ({100*valid_json_count/total_evaluated:.0f}%)")
print(f"   Category accuracy:    {correct_category}/{total_evaluated} ({100*correct_category/total_evaluated:.0f}%)")
print("=" * 70)

In [ ]:
# ========================================================================
# SHOW A DETAILED EXAMPLE WITH THE IMAGE
# ========================================================================
import matplotlib.pyplot as plt
from PIL import Image

# Pick a sample from validation set
demo_idx = 0
demo_sample = val_samples[demo_idx]
demo_image = demo_sample["messages"][1]["content"][0]["image"]
demo_expected = json.loads(demo_sample["messages"][2]["content"][0]["text"])

# Run prediction
demo_prediction = predict_pod(demo_image)

# Display
fig, axes = plt.subplots(1, 2, figsize=(16, 8))

# Show image
axes[0].imshow(demo_image)
axes[0].set_title("POD Image", fontsize=14)
axes[0].axis("off")

# Show JSON comparison
comparison_text = "EXPECTED:\n" + json.dumps(demo_expected, indent=2)[:500]
comparison_text += "\n\nPREDICTED:\n" + json.dumps(demo_prediction, indent=2)[:500]

axes[1].text(0.05, 0.95, comparison_text, transform=axes[1].transAxes,
             fontsize=8, verticalalignment="top", fontfamily="monospace",
             bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))
axes[1].set_title("Expected vs Predicted JSON", fontsize=14)
axes[1].axis("off")

plt.tight_layout()
plt.savefig("/kaggle/working/evaluation_demo.png", dpi=150)
plt.show()

---
## 💾 Step 7: Save Model

We save the model in multiple formats:
1. **LoRA adapter only** (~200MB) — lightweight, needs base model to use
2. **Merged 16-bit** (~14GB) — standalone, plug into any HF pipeline
3. **GGUF Q4_K_M** (~4.5GB) — for Ollama/llama.cpp local inference
4. **Push to HuggingFace Hub** — share publicly or privately

In [ ]:
# ========================================================================
# SAVE LoRA ADAPTER (lightweight — ~200MB)
# ========================================================================
LORA_DIR = "/kaggle/working/pod_qwen_lora"

model.save_pretrained(LORA_DIR)
tokenizer.save_pretrained(LORA_DIR)

print(f"✅ LoRA adapter saved to: {LORA_DIR}")
# Show size
import subprocess
result = subprocess.run(["du", "-sh", LORA_DIR], capture_output=True, text=True)
print(f"   Size: {result.stdout.strip()}")

In [ ]:
# ========================================================================
# SAVE MERGED 16-bit MODEL (full standalone model)
# NOTE: This requires ~14GB disk + RAM. Skip if disk space is limited.
# ========================================================================
MERGED_DIR = "/kaggle/working/pod_qwen_merged_16bit"

try:
    model.save_pretrained_merged(
        MERGED_DIR,
        tokenizer,
        save_method="merged_16bit",
    )
    print(f"✅ Merged 16-bit model saved to: {MERGED_DIR}")
except Exception as e:
    print(f"⚠️ Could not save merged model (might be disk/memory limited): {e}")
    print("   The LoRA adapter was still saved successfully above.")

In [ ]:
# ========================================================================
# (OPTIONAL) PUSH TO HUGGING FACE HUB
# ========================================================================
# To use this:
# 1. Go to Kaggle Settings → Secrets
# 2. Add a secret named "HF_TOKEN" with your HuggingFace token (Write access)
# 3. Uncomment and run this cell

PUSH_TO_HUB = False     # <-- Set to True to push
HUB_REPO = "your-username/qwen2.5-vl-7b-pod-analyzer"  # <-- Change this!

if PUSH_TO_HUB:
    from kaggle_secrets import UserSecretsClient
    try:
        secrets = UserSecretsClient()
        hf_token = secrets.get_secret("HF_TOKEN")
    except:
        hf_token = None
        print("⚠️ HF_TOKEN not found in Kaggle Secrets. Add it first!")
    
    if hf_token:
        print(f"📤 Pushing LoRA adapter to: {HUB_REPO}")
        model.push_to_hub_merged(
            HUB_REPO,
            tokenizer,
            save_method="lora",
            token=hf_token,
            private=True,
        )
        print(f"✅ Model pushed to: https://huggingface.co/{HUB_REPO}")
else:
    print("ℹ️ Skipping Hub push. Set PUSH_TO_HUB = True to enable.")

---
## 🔮 Step 8: Production Inference Example

Copy this code to use your fine-tuned model anywhere.

In [ ]:
# ========================================================================
# PRODUCTION INFERENCE TEMPLATE
# Copy this to your production code / API server
# ========================================================================

INFERENCE_CODE = '''
# =====================================================================
# PRODUCTION INFERENCE — Qwen2.5-VL-7B POD Analyzer
# =====================================================================
# Prerequisites:
#   pip install unsloth qwen-vl-utils torch
# =====================================================================

from unsloth import FastVisionModel
from qwen_vl_utils import process_vision_info
from PIL import Image
import torch, json

# --- Load model ---
# Option A: Load LoRA adapter (needs base model + adapter)
model, tokenizer = FastVisionModel.from_pretrained(
    model_name="path/to/pod_qwen_lora",   # or HuggingFace repo
    max_seq_length=2048,
    load_in_4bit=True,
)
FastVisionModel.for_inference(model)

# --- System prompt ---
SYSTEM_PROMPT = """You are a highly accurate document analysis AI specialized in processing Proof of Delivery (POD) images.
You must analyze the provided POD image and extract specific information.

Classification Rules (in priority order):
1. Physical Paper Damage -> MANUAL_CHECK_REQUIRED
2. Damage + Short in remarks -> ISSUE_POD_DAMAGED_AND_SHORT
3. Damage in remarks -> ISSUE_POD_DAMAGED
4. Shortage in remarks -> ISSUE_POD_SHORT
5. Seal/Stamp + Signature -> CLEAN_POD_SEAL_AND_SIGNATURE
6. Seal/Stamp Only -> CLEAN_POD_ONLY_SEAL
7. Signature Only -> CLEAN_POD_ONLY_SIGNATURE
8. Neither -> NO_SIGNATURE_NO_STAMP

Output ONLY valid JSON."""

# --- Predict ---
def analyze_pod(image_path: str) -> dict:
    messages = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {"role": "user", "content": [
            {"type": "image", "image": image_path},
            {"type": "text", "text": "Extract all POD fields into JSON."},
        ]},
    ]

    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = tokenizer(text=[text], images=image_inputs, videos=video_inputs,
                       padding=True, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output_ids = model.generate(**inputs, max_new_tokens=512, temperature=0.1, do_sample=True)

    generated = output_ids[:, inputs["input_ids"].shape[1]:]
    response = tokenizer.batch_decode(generated, skip_special_tokens=True)[0]

    clean = response.strip()
    for prefix in ["```json", "```"]: clean = clean.removeprefix(prefix)
    clean = clean.removesuffix("```").strip()
    return json.loads(clean)

# --- Usage ---
result = analyze_pod("path/to/pod_image.jpg")
print(json.dumps(result, indent=2))
'''

# Save the inference template
with open("/kaggle/working/inference_template.py", "w") as f:
    f.write(INFERENCE_CODE)

print("📄 Production inference template saved to: /kaggle/working/inference_template.py")
print("\nExpected output format:")
example_output = {
    "cnNumber": "2104261013196.0",
    "hasSignature": True,
    "hasStamp": False,
    "hasHandwriting": True,
    "imageQualityPassed": True,
    "remarksText": "Paint By/Bill. SNHA 2104",
    "deliveryDate": "2026-07-19",
    "categoryReason": "The document contains a handwritten signature but lacks a stamp.",
    "confidenceScore": 0.95,
    "podCategory": "CLEAN_POD_ONLY_SIGNATURE",
    "limit_exceed": False
}
print(json.dumps(example_output, indent=2))

---
## 📋 Step 9: Download Artifacts

After training, download these from `/kaggle/working/`:

| File/Folder | Description | Size |
|---|---|---|
| `pod_qwen_lora/` | LoRA adapter weights | ~200MB |
| `pod_qwen_merged_16bit/` | Full merged model | ~14GB |
| `training_loss.png` | Loss curve plot | ~100KB |
| `evaluation_demo.png` | Example prediction | ~200KB |
| `inference_template.py` | Production code | ~3KB |

### How to use the LoRA adapter later:
```python
from unsloth import FastVisionModel

model, tokenizer = FastVisionModel.from_pretrained(
    model_name="path/to/pod_qwen_lora",
    max_seq_length=2048,
    load_in_4bit=True,
)
FastVisionModel.for_inference(model)
```

In [ ]:
# ========================================================================
# FINAL SUMMARY
# ========================================================================
import os

print("\n" + "=" * 70)
print("🎉 FINE-TUNING COMPLETE — SUMMARY")
print("=" * 70)
print(f"\n📦 Base model:       {MODEL_NAME}")
print(f"📊 Training samples: {len(train_samples)}")
print(f"📊 Val samples:      {len(val_samples)}")
print(f"🔁 Epochs:           {NUM_EPOCHS}")
print(f"📉 Final train loss: {train_result.training_loss:.4f}")

if eval_results:
    print(f"\n🧪 Evaluation Results:")
    print(f"   Valid JSON:       {valid_json_count}/{total_evaluated} ({100*valid_json_count/total_evaluated:.0f}%)")
    print(f"   Category match:   {correct_category}/{total_evaluated} ({100*correct_category/total_evaluated:.0f}%)")

print(f"\n💾 Saved artifacts:")
for item in ["pod_qwen_lora", "pod_qwen_merged_16bit", "training_loss.png", "evaluation_demo.png", "inference_template.py"]:
    path = f"/kaggle/working/{item}"
    if os.path.exists(path):
        if os.path.isdir(path):
            size = sum(os.path.getsize(os.path.join(dp, f)) for dp, dn, filenames in os.walk(path) for f in filenames)
        else:
            size = os.path.getsize(path)
        print(f"   ✅ {item:<35s} {size/1e6:.1f} MB")
    else:
        print(f"   ⬜ {item:<35s} (not saved)")

print(f"\n📖 Next steps:")
print(f"   1. Download the LoRA adapter from /kaggle/working/pod_qwen_lora/")
print(f"   2. Use inference_template.py for production deployment")
print(f"   3. Or push to HuggingFace Hub (set PUSH_TO_HUB = True above)")
print("=" * 70)